# Step 6: generate MBBQ model outputs on Colab

Runs `my_research/generation/run_step6.py` for one model over en, nl, es, tr x {main, control}, with the paper-era generation settings from `my_research/generation/ask_model_paper.py` (the authors' `models.py` at commit `20ae39b`): Mistral gets 1000 new tokens and batch size 16, Zephyr 256 and 32, each with its original prompt format and a pinned model revision.
Everything is written to Google Drive, and finished subsets are skipped, so **after a disconnect just run all cells again**.

Before the first run:
1. Runtime > Change runtime type > **A100** (L4 also works). A T4 lacks bf16 and the script refuses it.
2. Optional: add a Colab secret (key icon on the left) named `HF_TOKEN` holding a read token, with notebook access on. Neither model is gated; a token only avoids anonymous rate limits.
3. Push `my_research/` to the fork (cell 2 clones it from GitHub).

Outputs go to `MyDrive/mbbq_paper`. Earlier runs in `MyDrive/mbbq` used upstream `ask_model` (100 new tokens for every model) and are not reused; the script stops if it finds outputs made with other settings in its folders. You can delete `MyDrive/mbbq`.

In [ ]:
# Which model: run once with "mistral", then once with "zephyr".
# Model id, pinned revision, prompt format, max_new_tokens and batch size
# come from PAPER_MODELS in my_research/generation/ask_model_paper.py.
MODEL_KEY = "mistral"
DRIVE = "/content/drive/MyDrive/mbbq_paper"
REPO = "https://github.com/michaellarionov/MBBQ_Russian.git"
TRANSFORMERS_PIN = "4.39.3"  # requirements.txt version; "" = Colab's own

In [ ]:
import os
from google.colab import drive, userdata
drive.mount("/content/drive")
try:  # optional; mbbq.py's login is skipped and the hub reads HF_TOKEN
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except userdata.SecretNotFoundError:
    pass
if not os.path.isdir("/content/MBBQ_Russian"):
    !git clone -q {REPO} /content/MBBQ_Russian
%cd /content/MBBQ_Russian
!git pull -q && git log --oneline -1
if TRANSFORMERS_PIN:
    !pip install -q transformers=={TRANSFORMERS_PIN}
!python -c "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.get_device_name())"

## Smoke test (do this first)

One subset (Gender_identity, English, main set: 2,720 prompts). It checks that loading, generation and answer detection work, and it estimates the GPU time for the full run. Mistral's 1000-token limit makes it much slower per prompt than upstream's 100.

In [ ]:
!python my_research/generation/run_step6.py --model {MODEL_KEY} --key {MODEL_KEY}_smoke \
    --langs en --subsets Gender_identity --no_control \
    --work_dir {DRIVE}/work --results_dir {DRIVE}/smoke
# `!` does not stop "Run all" on failure; this does.
assert globals().get("_exit_code", 0) == 0, "run failed: see the error above"


In [ ]:
import json, sys
sys.path.insert(0, "my_research/scorer")
from mbbq_scorer import annotate, attach_polarity, load_outputs, score

smoke_pkl = f"{DRIVE}/smoke/{MODEL_KEY}_smoke_en.pkl"
assert os.path.exists(smoke_pkl), "no smoke output: the run in the cell above failed"
smoke = load_outputs(smoke_pkl)
print(smoke[["prompt_id", "answer", "answer_detected"]].sample(5, random_state=0).to_string())
s = score(annotate(attach_polarity(smoke, "data", "en")))
print({k: round(v, 3) for k, v in s.items()})

info = json.load(open(f"{DRIVE}/smoke/{MODEL_KEY}_smoke_runinfo.json"))
secs = sum(info["ask_seconds"].values())
full = 8 * 50_400  # prompts: 4 languages x {main, control}
print(f"{info['gpu']}: {secs / len(smoke):.3f} s/prompt -> "
      f"~{secs / len(smoke) * full / 3600:.0f} GPU-hours for the full run")

Continue only if the answers look sensible, few are undetected, and the GPU-hour estimate fits your compute units.

## Full run

This takes many hours. If Colab disconnects, run all cells again, skipping the smoke-test cells; it resumes at the unfinished subset.

In [ ]:
!python my_research/generation/run_step6.py --model {MODEL_KEY} \
    --work_dir {DRIVE}/work --results_dir {DRIVE}/results
# `!` does not stop "Run all" on failure; this does.
assert globals().get("_exit_code", 0) == 0, "run failed: see the error above"


In [ ]:
info = json.load(open(f"{DRIVE}/results/{MODEL_KEY}_runinfo.json"))
print(json.dumps({k: v for k, v in info.items() if k != "ask_seconds"}, indent=1))
print(f"GPU time so far: {sum(info['ask_seconds'].values()) / 3600:.1f} h")

When all 8 `<model>_<lang>[_control].pkl` files and `<model>_runinfo.json` are in `MyDrive/mbbq_paper/results`, download them into `my_research/results/` locally (about 50 MB per pickle) and run step 7. The runinfo records the generation settings (model revision, prompt format, max_new_tokens, batch size) and library versions.